# Group 8 — International Finance for Clean Energy
AS5004 Foundational Statistics for AI · MCA Semester 1 · Unit 2

The group handout calls the enhanced visualisation/SDG task Phase 5; in the repository plan it is Phase 4. Repository Phase 5 is the separate country-held-out machine-learning task.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Change this value to repeat the country-grouped model test with another seed.
RANDOM_STATE = 42

ROOT = Path.cwd()
if not (ROOT / 'data' / 'Group_08_Dataset.xlsx').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'Group_08_Dataset.xlsx'
OUT = ROOT / 'outputs'; FIG = OUT / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_excel(DATA, sheet_name='Data', header=11)
numeric = ['year','clean_energy_finance_all_usd_m','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m','elec_access_total_pct','renewable_share_tfec_pct','re_capacity_all_w_per_capita']
for c in numeric: df[c] = pd.to_numeric(df[c], errors='coerce')
print(df.shape, 'rows and columns')
print(df.country.nunique(), 'countries;', df.year.min(), 'to', df.year.max())
display(df.head())

## Phase 1 — Data quality audit
The workbook specifies row 12 as the header. Blank cells are retained as missing, not recoded to zero. The raw workbook remains unchanged.

In [ ]:
audit = pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_count':df.isna().sum(),'missing_percent':(df.isna().mean()*100).round(2),'unique_values':df.nunique()})
display(audit)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate country-year keys:', df.duplicated(['iso3','year']).sum())
OUT.mkdir(exist_ok=True); audit.to_csv(OUT/'data_quality_audit.csv')

## Phase 2 — Descriptive statistics
Compute mean, median, mode, variance, standard deviation, IQR, skewness and kurtosis. Compare by continent.

In [ ]:
key = ['clean_energy_finance_all_usd_m','elec_access_total_pct','renewable_share_tfec_pct','re_capacity_all_w_per_capita','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m']
def stats(s):
    s=s.dropna(); q1,q3=s.quantile([.25,.75]); modes=s.mode()
    return pd.Series({'n':len(s),'mean':s.mean(),'median':s.median(),'mode':', '.join(map(str,modes.head(3).tolist())),'variance':s.var(),'sd':s.std(),'Q1':q1,'Q3':q3,'IQR':q3-q1,'skewness':s.skew(),'kurtosis':s.kurt()})
summary=pd.DataFrame({c:stats(df[c]) for c in key}).T
display(summary); summary.to_csv(OUT/'descriptive_statistics.csv')
grouped=df.groupby('continent')[key[:4]].agg(['count','mean','median'])
display(grouped); grouped.to_csv(OUT/'continent_summary.csv')

## Phase 3 — Zero values and outliers
Z-score flags use |z| > 3. IQR flags use 1.5 × IQR fences. Outliers are not automatically deleted; large financial flows may be substantively important.

In [ ]:
finance_cols=['clean_energy_finance_all_usd_m','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m']
zero_skew=pd.DataFrame({'n':df[finance_cols].count(),'zeros':(df[finance_cols]==0).sum(),'zero_percent':(df[finance_cols].eq(0).sum()/df[finance_cols].count()*100),'skewness':df[finance_cols].skew()})
display(zero_skew); zero_skew.to_csv(OUT/'finance_zero_skewness.csv')
def outlier_report(col):
    s=df[col].dropna(); z=(s-s.mean())/s.std(); q1,q3=s.quantile([.25,.75]); iqr=q3-q1; zf=z.abs()>3; qf=(s<q1-1.5*iqr)|(s>q3+1.5*iqr)
    return {'n':len(s),'zscore_count':int(zf.sum()),'zscore_percent':float(zf.mean()*100),'iqr_low':float(q1-1.5*iqr),'iqr_high':float(q3+1.5*iqr),'iqr_count':int(qf.sum()),'iqr_percent':float(qf.mean()*100),'both':int((zf&qf).sum())}
outliers=pd.DataFrame({c:outlier_report(c) for c in ['clean_energy_finance_all_usd_m','re_capacity_all_w_per_capita']}).T
display(outliers); outliers.to_csv(OUT/'outlier_results.csv')
top=df.nlargest(15,'clean_energy_finance_all_usd_m')[['country','iso3','continent','year','clean_energy_finance_all_usd_m']]
display(top); top.to_csv(OUT/'top_finance_country_years.csv',index=False)

# Export row-level flags for review. Z-score uses sample SD; IQR uses 1.5 x IQR.
flag_rows = []
flag_summary_rows = []
for col in ["clean_energy_finance_all_usd_m", "re_capacity_all_w_per_capita"]:
    s = df[col].dropna()
    z_scores = (s - s.mean()) / s.std(ddof=1)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    z_flag = z_scores.abs() > 3
    iqr_flag = (s < lower_fence) | (s > upper_fence)
    flagged = df.loc[s.index, ["country", "iso3", "continent", "year", col]].copy()
    flagged["z_score"] = z_scores
    flagged["z_score_flag"] = z_flag
    flagged["iqr_flag"] = iqr_flag
    flagged["iqr_lower_fence"] = lower_fence
    flagged["iqr_upper_fence"] = upper_fence
    flagged["variable"] = col
    flag_rows.append(flagged.loc[z_flag | iqr_flag])
    flag_summary_rows.append({
        "variable": col, "n": len(s), "z_score_flag_count": int(z_flag.sum()),
        "iqr_flag_count": int(iqr_flag.sum()),
        "both_methods_count": int((z_flag & iqr_flag).sum()),
        "z_score_only_count": int((z_flag & ~iqr_flag).sum()),
        "iqr_only_count": int((iqr_flag & ~z_flag).sum()),
        "iqr_lower_fence": float(lower_fence), "iqr_upper_fence": float(upper_fence),
    })
outlier_method_comparison = pd.DataFrame(flag_summary_rows)
outlier_flagged_country_years = pd.concat(flag_rows, ignore_index=True)
outlier_method_comparison.to_csv(OUT / "outlier_method_comparison.csv", index=False)
outlier_flagged_country_years.to_csv(OUT / "outlier_flagged_country_years.csv", index=False)
display(outlier_method_comparison)
print("Flagged country-year-variable rows exported:", len(outlier_flagged_country_years))


## Phase 4 — Visualisations and SDG 7.a interpretation
Histograms and boxplots are generated for key variables. Finance/capacity use log1p transforms so zero observations remain valid.

In [ ]:
plot_cols=key
for c in plot_cols:
    s=df[c].dropna(); fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    if 'finance' in c or 'capacity' in c: vals=np.log1p(s.clip(lower=0)); xlabel='log1p(value)'
    else: vals=s; xlabel=c
    ax[0].hist(vals,bins=35,edgecolor='black'); ax[0].set_xlabel(xlabel); ax[0].set_ylabel('Country-years'); ax[0].set_title('Histogram: '+c)
    ax[1].boxplot(vals); ax[1].set_ylabel(xlabel); ax[1].set_title('Boxplot: '+c)
    fig.tight_layout(); fig.savefig(FIG/(c+'_distribution.png'),dpi=150); plt.show()
df['decade']=(df.year//10*10).astype(int).astype(str)+'s'
gd=df.groupby(['continent','decade'])['clean_energy_finance_all_usd_m'].median().reset_index()
fig,ax=plt.subplots(figsize=(9,4))
for name,g in gd.groupby('continent'): ax.plot(g.decade,g.clean_energy_finance_all_usd_m,marker='o',label=name)
ax.set_yscale('symlog',linthresh=1); ax.set_xlabel('Decade'); ax.set_ylabel('Median finance (million constant 2023 US$)'); ax.set_title('Median finance by continent and decade'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/'finance_by_continent_decade.png',dpi=150); plt.show()
country_totals=df.groupby(['country','iso3','continent'])['clean_energy_finance_all_usd_m'].sum().sort_values(ascending=False).reset_index()
display(country_totals.head(15)); country_totals.to_csv(OUT/'country_finance_totals.csv',index=False)

## Group Task Phase 5 — Visualisations and SDG 7.a interpretation

**Repository phase mapping:** this assignment task is **Phase 4** in `docs/PROJECT_PLAN.md`. The repository's **Phase 5** is the separate machine-learning task. Both labels are retained to match the handout and repository checklist.

This section adds the top-10 country/area finance ranking and finance/renewable-share association while preserving the existing histograms, boxplots, continent trend and country finance summary. It uses pairwise complete observations; missing values are not recoded as zero. The Data Dictionary controls SDG indicator mappings.


In [ ]:
# Country cumulative finance ranking and observation coverage
FINANCE = "clean_energy_finance_all_usd_m"
RENEWABLE_SHARE = "renewable_share_tfec_pct"
country_finance = (df.groupby(["country", "iso3", "continent"], dropna=False)
    .agg(total_finance_musd=(FINANCE, "sum"), observed_years=("year", "nunique"),
         first_year=("year", "min"), last_year=("year", "max"),
         finance_observations=(FINANCE, "count"))
    .reset_index().sort_values("total_finance_musd", ascending=False))
country_finance.to_csv(OUT / "country_finance_totals.csv", index=False)
top10 = country_finance.head(10).copy()
top10.to_csv(OUT / "top_10_countries_finance.csv", index=False)
plot_top10 = top10.sort_values("total_finance_musd").copy()
plot_top10["chart_country"] = plot_top10["country"].replace({"Lao People's Democratic Republic": "Lao PDR"})
fig, ax = plt.subplots(figsize=(10.5, 6.5))
ax.barh(plot_top10["chart_country"], plot_top10["total_finance_musd"])
max_value = top10["total_finance_musd"].max()
for pos, (_, row) in enumerate(plot_top10.iterrows()):
    ax.text(row["total_finance_musd"] + max_value * .012, pos,
            f'{row["total_finance_musd"]:,.0f} | {int(row["observed_years"])} years',
            va="center", fontsize=8)
ax.set_xlabel("Cumulative clean-energy finance, 2010–2024 (million constant 2023 US$)")
ax.set_ylabel("Country or area")
ax.set_title("Top 10 countries/areas by cumulative clean-energy finance, 2010–2024")
ax.set_xlim(0, max_value * 1.32)
fig.tight_layout()
fig.savefig(FIG / "top_10_countries_finance.png", dpi=250, bbox_inches="tight")
plt.show()

# Continent totals and typical country-year finance; both are needed for comparison.
continent_finance = (df.groupby("continent")[FINANCE]
    .agg(country_years="count", total_finance_musd="sum",
         mean_finance_musd="mean", median_finance_musd="median")
    .sort_values("total_finance_musd", ascending=False))
continent_finance.to_csv(OUT / "continent_finance_summary.csv")
print(continent_finance)

# Pairwise complete finance/share observations. log1p retains zero finance values.
pair = df[[FINANCE, RENEWABLE_SHARE, "country", "iso3", "continent", "year"]].dropna().copy()
pair = pair[(pair[FINANCE] >= 0) & pair[RENEWABLE_SHARE].between(0, 100)]
pair["log1p_finance"] = np.log1p(pair[FINANCE])
pearson_raw = pair[FINANCE].corr(pair[RENEWABLE_SHARE])
pearson_log = pair["log1p_finance"].corr(pair[RENEWABLE_SHARE])
spearman_raw = pair[FINANCE].corr(pair[RENEWABLE_SHARE], method="spearman")
correlations = pd.DataFrame([{
    "pairwise_country_years": len(pair),
    "pearson_raw_finance": pearson_raw,
    "pearson_log1p_finance": pearson_log,
    "spearman_raw_finance": spearman_raw,
}])
correlations.to_csv(OUT / "finance_vs_renewable_share_correlation.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 6.5))
for continent, group in pair.groupby("continent"):
    ax.scatter(group["log1p_finance"], group[RENEWABLE_SHARE],
               s=18, alpha=.5, label=continent)
trend_x = np.linspace(pair["log1p_finance"].min(), pair["log1p_finance"].max(), 100)
slope, intercept = np.polyfit(pair["log1p_finance"], pair[RENEWABLE_SHARE], 1)
ax.plot(trend_x, intercept + slope * trend_x, linewidth=2, label="Pooled linear trend")
ax.set_xlabel("log1p(clean-energy finance; million constant 2023 US$)")
ax.set_ylabel("Renewable share of total final energy consumption (%)")
ax.set_title("Clean-energy finance versus renewable energy share")
ax.text(.02, .97,
        f"Pairwise observations: {len(pair):,}\nPearson r (raw finance): {pearson_raw:.3f}\nPearson r (log1p finance): {pearson_log:.3f}\nSpearman rho (raw finance): {spearman_raw:.3f}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={"boxstyle": "round,pad=.4", "facecolor": "white", "alpha": .9})
ax.legend(title="Continent", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIG / "finance_vs_renewable_share.png", dpi=250, bbox_inches="tight")
plt.show()
print(correlations)


## Repository Phase 5 — Machine-learning classification

**Target:** renewable energy share; high means strictly above the renewable-share median calculated from training observations only. Split country groups first, then calculate the threshold and apply it unchanged to both training and test rows.

**Predictors:** log1p finance, electricity access, log1p installed renewable capacity and year. Complete cases are used for the target and predictors.

**Validation:** country-grouped `GroupShuffleSplit`; no country/area may occur in both sets. Compare logistic regression with a majority-class baseline using accuracy, precision, recall and a confusion matrix.


In [ ]:
target='renewable_share_tfec_pct'
raw_features=['clean_energy_finance_all_usd_m','elec_access_total_pct','re_capacity_all_w_per_capita','year']
m=df[['country','iso3',target]+raw_features].dropna().copy()
m=m[(m.clean_energy_finance_all_usd_m>=0)&(m.re_capacity_all_w_per_capita>=0)].copy()

# Split by country FIRST. GroupShuffleSplit does not use target labels.
features=['log_finance','elec_access_total_pct','log_capacity','year']
m['log_finance']=np.log1p(m.clean_energy_finance_all_usd_m)
m['log_capacity']=np.log1p(m.re_capacity_all_w_per_capita)
split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=RANDOM_STATE)
train,test=next(split.split(m[raw_features],groups=m.iso3))
train_countries=set(m.iloc[train].iso3)
test_countries=set(m.iloc[test].iso3)
assert train_countries.isdisjoint(test_countries), 'Country leakage detected'

# Calculate the target cutoff on training rows only; apply the same cutoff to test rows.
median_target=m.iloc[train][target].median()
m['label']=(m[target]>median_target).astype(int)
X=m[features]
y=m.label
X_train,X_test=X.iloc[train],X.iloc[test]
y_train,y_test=y.iloc[train],y.iloc[test]
assert len(train)+len(test)==len(m)
assert train_countries.isdisjoint(test_countries)

print('Complete rows:',len(m),'Countries:',m.iso3.nunique())
print('Training-only renewable-share median threshold (%):',round(median_target,2))
print('Train rows/countries:',len(train),len(train_countries))
print('Test rows/countries:',len(test),len(test_countries))
print('Country overlap:',len(train_countries & test_countries))
print('Training class proportions:')
display(y_train.value_counts(normalize=True).sort_index().rename('proportion').to_frame())
print('Test class proportions using the same training threshold:')
display(y_test.value_counts(normalize=True).sort_index().rename('proportion').to_frame())
pd.DataFrame([{'train_rows':len(train),'test_rows':len(test),
 'train_countries':len(train_countries),'test_countries':len(test_countries),
 'training_median_threshold_pct':median_target,
 'country_overlap':len(train_countries & test_countries)}]).to_csv(OUT/'model_target_threshold.csv',index=False)

model=Pipeline([('scale',StandardScaler()),('logistic',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42))])
model.fit(X_train,y_train)
pred=model.predict(X_test)
base=np.full(len(y_test),y_train.mode().iloc[0])
metrics=pd.DataFrame([
 {'model':'Majority baseline','accuracy':accuracy_score(y_test,base),'precision':precision_score(y_test,base,zero_division=0),'recall':recall_score(y_test,base,zero_division=0)},
 {'model':'Logistic regression','accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0)}
])
display(metrics)
metrics.to_csv(OUT/'model_metrics.csv',index=False)
print(classification_report(y_test,pred,target_names=['low','high'],zero_division=0))
cm=confusion_matrix(y_test,pred,labels=[0,1])
ConfusionMatrixDisplay(cm,display_labels=['Low','High']).plot(values_format='d')
plt.title('Confusion matrix — held-out countries')
plt.tight_layout()
plt.savefig(FIG/'confusion_matrix.png',dpi=150,bbox_inches='tight')
plt.show()
evaluation=m.iloc[test][['country','iso3','year',target,'label']].copy()
evaluation['predicted']=pred
evaluation['correct']=evaluation.label==evaluation.predicted
evaluation.to_csv(OUT/'heldout_predictions.csv',index=False)
error_summary=evaluation.loc[~evaluation.correct].groupby(['country','year']).size().reset_index(name='incorrect_rows')
error_summary.to_csv(OUT/'heldout_errors_by_country_year.csv',index=False)


## Limitations and conclusion
The class threshold is training-only, preventing the test-target distribution from influencing class construction. Discuss missing-data reduction, modelled UN estimates, percentage values clustered near 100, the limitations of a single grouped split, and the fact that association is not causation. Report whether the model meaningfully beats the baseline. Cite UN DESA SDG Global Database and the custodian agencies named in the workbook Data Dictionary. Acknowledge AI assistance as required by your handout.

## Validation — Group Task Phase 5 outputs

The following checks verify the top-10 ranking, correlation calculations, row-level outlier flags, country separation and generated outputs. This is the group's visualisation assignment (repository Phase 4), not the repository's ML Phase 5.



In [ ]:
required_paths = [
    OUT / "top_10_countries_finance.csv",
    OUT / "finance_vs_renewable_share_correlation.csv",
    OUT / "outlier_method_comparison.csv",
    OUT / "outlier_flagged_country_years.csv",
    OUT / "model_target_threshold.csv",
    OUT / "model_metrics.csv",
    FIG / "top_10_countries_finance.png",
    FIG / "finance_vs_renewable_share.png",
]
missing_paths = [str(p.relative_to(OUT.parent)) for p in required_paths if not p.exists()]
assert not missing_paths, f"Missing expected outputs: {missing_paths}"
assert len(top10) == 10, "Top 10 output should contain 10 country/areas"
assert len(pair) == int(correlations.loc[0, "pairwise_country_years"])
assert train_countries.isdisjoint(test_countries), "Country leakage detected"
assert np.isclose(median_target, m.iloc[train][target].median()), "Threshold is not training-derived"
assert len(outlier_flagged_country_years) > 0, "Expected outlier detail rows"
validation = pd.DataFrame([{
    "top10_rows": len(top10),
    "finance_share_pairwise_n": len(pair),
    "pearson_raw": pearson_raw,
    "pearson_log1p": pearson_log,
    "spearman_raw": spearman_raw,
    "outlier_flagged_rows": len(outlier_flagged_country_years),
    "train_countries": len(train_countries),
    "test_countries": len(test_countries),
    "country_overlap": len(train_countries & test_countries),
    "train_only_target_median": median_target,
    "required_files_present": len(required_paths),
    "missing_files": len(missing_paths),
}])
validation.to_csv(OUT / "final_validation.csv", index=False)
display(validation)
print("Notebook validation passed.")



In [ ]:
print("Top 10 countries/areas by cumulative finance")
display(top10[["country", "total_finance_musd", "observed_years", "first_year", "last_year"]])
print("\\nFinance and renewable-share correlations (pairwise complete observations)")
display(correlations)
print("\\nML threshold and baseline comparison")
display(pd.read_csv(OUT / "model_target_threshold.csv"))
display(metrics)



## SDG indicator mapping and interpretation

The workbook's Data Dictionary maps `clean_energy_finance_all_usd_m` and its technology breakdowns to **SDG 7.a.1** (Target 7.a). Electricity access is **7.1.1** (Target 7.1), renewable share of total final energy consumption is **7.2.1** (Target 7.2), and installed renewable electricity capacity per capita is **7.b.1** (Target 7.b). These supporting indicators are not substitutes for the direct 7.a.1 financial-flow measure. Pooled correlation is descriptive and does not establish causation.


## Reproducibility and limitations

Run the notebook from the repository root with the instructor-provided workbook placed locally in `data/Group_08_Dataset.xlsx`. The workbook must stay within Group 8, as directed in its README; it is intentionally not public. The visualisation task is Group Task Phase 5 in the handout and Repository Phase 4 in the checklist. Repository Phase 5 is the separate ML task. The country-grouped test split and training-only median avoid country leakage and test-distribution threshold leakage, but a single split has sampling variability. Association is not causation; UN indicator values may be modelled estimates. A second clean-environment run by another group member is still recommended.



## Figure gallery — Group Task Phase 5 (Repository Phase 4)

The charts below are generated by the notebook and committed to the repository.

![Top 10 countries by cumulative clean-energy finance](../outputs/figures/top_10_countries_finance.png)

![Finance versus renewable energy share](../outputs/figures/finance_vs_renewable_share.png)

The top-10 chart ranks cumulative 2010–2024 finance and labels each country's 15 observed years. The scatter plot uses pairwise complete observations and a log1p finance x-axis to retain zero-finance observations while reducing skew.


## Repository Phase 5 — Machine-learning results and error review

The target threshold is calculated **after the country-grouped split**, using training observations only. The training median is **25.01%**. It is applied unchanged to training and test rows. The split contains **1,578 training rows from 110 countries** and **533 test rows from 37 countries**, with **zero country overlap**.

| Metric | Majority-class baseline | Logistic regression |
|---|---:|---:|
| Accuracy | 39.02% | 75.98% |
| Precision (high class) | 0.00% | 87.45% |
| Recall (high class) | 0.00% | 70.77% |

Logistic regression correctly classifies **405 of 533** test observations. Its confusion matrix is **TN=175, FP=33, FN=95, TP=230**. It makes 128 errors, with at least one error in 19 of the 37 test countries. The baseline predicts the training-majority low class; the test sample contains 208 low and 325 high observations, so its accuracy falls to 39.02% under this class-distribution shift.

The model performs better than the baseline on this one deterministic split, but this does not establish stable generalisation. Repeated country-grouped splits or grouped cross-validation are the next testing step. These are predictive associations, not evidence that finance causes renewable-energy adoption.


## Repository Phase 5 — Repeated country-grouped validation

This section evaluates the classifier across ten repeated country-held-out splits and five-fold country-grouped cross-validation. The country identifier iso3 is the grouping key in both procedures.

- **Repeated testing:** GroupShuffleSplit, test_size=0.25, with random seeds **42–51**.
- **Cross-validation:** five-fold GroupKFold; each country/area is held out in exactly one fold.
- **Target:** within each split or fold, calculate the median of renewable_share_tfec_pct from the training observations only. Label values strictly above that training median as high, and apply the same threshold unchanged to the test observations.
- **Predictors:** log1p total clean-energy finance, electricity access, log1p installed renewable capacity, and year.
- **Model:** standard scaling followed by logistic regression with class_weight="balanced".
- **Baseline:** predict the majority class in the training partition; ties are resolved as low.
- **Metrics:** accuracy, precision, recall and F1. Precision, recall and F1 treat high as the positive class. Summary standard deviations use the sample standard deviation (ddof=1).

**Expected logistic-regression accuracy checks (mean ± sample SD):** ten repeated splits, **79.20% ± 2.88%**; five-fold GroupKFold, **79.52% ± 4.50%**.

The result tables are written to outputs/phase6_repeated_testing/. The repeated-split accuracy boxplot is written to outputs/figures/phase6_accuracy_boxplot.png.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

VALIDATION_TARGET = "renewable_share_tfec_pct"
VALIDATION_FINANCE = "clean_energy_finance_all_usd_m"
VALIDATION_ACCESS = "elec_access_total_pct"
VALIDATION_CAPACITY = "re_capacity_all_w_per_capita"
VALIDATION_FEATURES = ["log_finance", VALIDATION_ACCESS, "log_capacity", "year"]
VALIDATION_COLUMNS = [
    "country", "iso3", "continent", "year", VALIDATION_TARGET,
    VALIDATION_FINANCE, VALIDATION_ACCESS, VALIDATION_CAPACITY,
]

validation_data = df[VALIDATION_COLUMNS].copy()
for column in ["year", VALIDATION_TARGET, VALIDATION_FINANCE,
               VALIDATION_ACCESS, VALIDATION_CAPACITY]:
    validation_data[column] = pd.to_numeric(validation_data[column], errors="coerce")
validation_data = validation_data.dropna(subset=VALIDATION_COLUMNS).copy()
validation_data = validation_data[
    (validation_data[VALIDATION_FINANCE] >= 0)
    & (validation_data[VALIDATION_CAPACITY] >= 0)
].copy()
validation_data["log_finance"] = np.log1p(validation_data[VALIDATION_FINANCE])
validation_data["log_capacity"] = np.log1p(validation_data[VALIDATION_CAPACITY])
validation_data = validation_data.reset_index(drop=True)


def validation_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def evaluate_validation_partition(train_indices, test_indices, partition, kind):
    train = validation_data.iloc[train_indices]
    test = validation_data.iloc[test_indices]
    train_countries = set(train["iso3"])
    test_countries = set(test["iso3"])
    overlap = train_countries & test_countries
    assert not overlap, f"Country leakage in {kind} {partition}: {sorted(overlap)}"

    # Calculate the target cutoff from training rows only.
    threshold = float(train[VALIDATION_TARGET].median())
    y_train = (train[VALIDATION_TARGET].to_numpy() > threshold).astype(int)
    y_test = (test[VALIDATION_TARGET].to_numpy() > threshold).astype(int)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    )
    model.fit(train[VALIDATION_FEATURES], y_train)
    prediction = model.predict(test[VALIDATION_FEATURES]).astype(int)

    class_counts = np.bincount(y_train, minlength=2)
    baseline_class = 0 if class_counts[0] >= class_counts[1] else 1
    baseline_prediction = np.full(len(test), baseline_class, dtype=int)

    model_scores = validation_metrics(y_test, prediction)
    baseline_scores = validation_metrics(y_test, baseline_prediction)
    row = {
        "partition": partition,
        "threshold_train_median_pct": threshold,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_countries": len(train_countries),
        "test_countries": len(test_countries),
        "country_overlap": len(overlap),
        "train_low": int((y_train == 0).sum()),
        "train_high": int((y_train == 1).sum()),
        "test_low": int((y_test == 0).sum()),
        "test_high": int((y_test == 1).sum()),
        "baseline_tie": bool(class_counts[0] == class_counts[1]),
        "baseline_class": "low" if baseline_class == 0 else "high",
    }
    row.update({f"model_{key}": value for key, value in model_scores.items()})
    row.update({f"baseline_{key}": value for key, value in baseline_scores.items()})

    held_out = test[["country", "iso3", "continent", "year"]].copy()
    held_out["row_index"] = test.index
    held_out["partition"] = partition
    held_out["actual"] = y_test
    held_out["predicted"] = prediction
    return row, held_out


def validation_summary(metrics_frame):
    records = []
    for model_name, prefix in [
        ("Logistic regression", "model"),
        ("Majority baseline", "baseline"),
    ]:
        for metric in ["accuracy", "precision", "recall", "f1"]:
            values = metrics_frame[f"{prefix}_{metric}"].astype(float)
            records.append({
                "model": model_name,
                "metric": metric,
                "mean": values.mean(),
                "sd_sample": values.std(ddof=1),
                "min": values.min(),
                "max": values.max(),
            })
    return pd.DataFrame(records)


def continent_error_counts(predictions):
    errors = predictions.copy()
    errors["false_negative"] = (errors["actual"] == 1) & (errors["predicted"] == 0)
    errors["false_positive"] = (errors["actual"] == 0) & (errors["predicted"] == 1)
    errors["misclassified"] = errors["actual"] != errors["predicted"]
    result = errors.groupby("continent").agg(
        test_exposures=("actual", "size"),
        false_negatives=("false_negative", "sum"),
        false_positives=("false_positive", "sum"),
        misclassified=("misclassified", "sum"),
    ).reset_index()
    result["error_rate"] = result["misclassified"] / result["test_exposures"]
    result["net_high_underprediction_fn_minus_fp"] = (
        result["false_negatives"] - result["false_positives"]
    )
    return result.sort_values("misclassified", ascending=False)

In [ ]:
# Ten country-held-out repetitions, using random seeds 42 through 51.
repeat_rows = []
repeat_predictions = []
repeat_audits = []
for seed in range(42, 52):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    train_indices, test_indices = next(
        splitter.split(validation_data[VALIDATION_FEATURES], groups=validation_data["iso3"])
    )
    row, held_out = evaluate_validation_partition(
        train_indices, test_indices, seed, "repeat"
    )
    row["random_state"] = seed
    repeat_rows.append(row)
    repeat_predictions.append(held_out)
    repeat_audits.append({
        "random_state": seed,
        "train_rows": row["train_rows"],
        "test_rows": row["test_rows"],
        "train_countries": row["train_countries"],
        "test_countries": row["test_countries"],
        "country_overlap": row["country_overlap"],
        "threshold_train_median_pct": row["threshold_train_median_pct"],
        "baseline_tie": row["baseline_tie"],
        "baseline_class": row["baseline_class"],
    })

repeat_metrics = pd.DataFrame(repeat_rows)
repeat_held_out = pd.concat(repeat_predictions, ignore_index=True)
repeat_summary = validation_summary(repeat_metrics)

# Five-fold GroupKFold; each country/area is held out in one fold.
fold_rows = []
fold_predictions = []
fold_audits = []
group_kfold = GroupKFold(n_splits=5)
for fold_number, (train_indices, test_indices) in enumerate(
    group_kfold.split(validation_data[VALIDATION_FEATURES], groups=validation_data["iso3"]),
    start=1,
):
    row, held_out = evaluate_validation_partition(
        train_indices, test_indices, fold_number, "fold"
    )
    fold_rows.append(row)
    fold_predictions.append(held_out)
    fold_audits.append({
        "fold": fold_number,
        "train_rows": row["train_rows"],
        "test_rows": row["test_rows"],
        "train_countries": row["train_countries"],
        "test_countries": row["test_countries"],
        "country_overlap": row["country_overlap"],
        "threshold_train_median_pct": row["threshold_train_median_pct"],
        "baseline_tie": row["baseline_tie"],
        "baseline_class": row["baseline_class"],
    })

fold_metrics = pd.DataFrame(fold_rows)
fold_held_out = pd.concat(fold_predictions, ignore_index=True)
fold_summary = validation_summary(fold_metrics)
assert len(fold_held_out) == len(validation_data)
assert fold_held_out["row_index"].nunique() == len(validation_data)
assert fold_held_out["iso3"].nunique() == validation_data["iso3"].nunique()
assert fold_held_out.groupby("iso3")["partition"].nunique().max() == 1

# Check the reference accuracy mean and sample SD values.
repeat_accuracy_mean = repeat_metrics["model_accuracy"].mean() * 100
repeat_accuracy_sd = repeat_metrics["model_accuracy"].std(ddof=1) * 100
fold_accuracy_mean = fold_metrics["model_accuracy"].mean() * 100
fold_accuracy_sd = fold_metrics["model_accuracy"].std(ddof=1) * 100

assert (round(repeat_accuracy_mean, 2), round(repeat_accuracy_sd, 2)) == (79.20, 2.88)
assert (round(fold_accuracy_mean, 2), round(fold_accuracy_sd, 2)) == (79.52, 4.50)
assert repeat_metrics["country_overlap"].max() == 0
assert fold_metrics["country_overlap"].max() == 0

repeat_continent_errors = continent_error_counts(repeat_held_out)
fold_continent_errors = continent_error_counts(fold_held_out)

# Save metrics, fold audits and continent error counts using repository-relative paths.
VALIDATION_OUT = OUT / "phase6_repeated_testing"
VALIDATION_OUT.mkdir(parents=True, exist_ok=True)
repeat_metrics.to_csv(VALIDATION_OUT / "phase6_repeated_split_metrics.csv", index=False)
repeat_summary.to_csv(VALIDATION_OUT / "phase6_repeated_summary_statistics.csv", index=False)
pd.DataFrame(repeat_audits).to_csv(
    VALIDATION_OUT / "phase6_repeated_split_audit.csv", index=False
)
fold_metrics.to_csv(VALIDATION_OUT / "phase6_groupkfold_fold_metrics.csv", index=False)
fold_summary.to_csv(VALIDATION_OUT / "phase6_groupkfold_summary_statistics.csv", index=False)
pd.DataFrame(fold_audits).to_csv(
    VALIDATION_OUT / "phase6_groupkfold_fold_audit.csv", index=False
)
repeat_continent_errors.to_csv(
    VALIDATION_OUT / "phase6_repeated_continent_error_analysis.csv", index=False
)
fold_continent_errors.to_csv(
    VALIDATION_OUT / "phase6_groupkfold_continent_error_analysis.csv", index=False
)

# Display raw fold metrics and summaries as percentages.
repeat_columns = [
    "random_state", "threshold_train_median_pct", "model_accuracy",
    "model_precision", "model_recall", "model_f1", "baseline_accuracy",
]
repeat_display = repeat_metrics[repeat_columns].copy()
repeat_display.loc[:, repeat_columns[2:]] *= 100

fold_columns = [
    "partition", "threshold_train_median_pct", "model_accuracy",
    "model_precision", "model_recall", "model_f1", "baseline_accuracy",
    "country_overlap",
]
fold_display = fold_metrics[fold_columns].copy()
fold_display.loc[:, fold_columns[2:7]] *= 100

repeat_summary_display = repeat_summary.copy()
repeat_summary_display[["mean", "sd_sample", "min", "max"]] *= 100
fold_summary_display = fold_summary.copy()
fold_summary_display[["mean", "sd_sample", "min", "max"]] *= 100

print("Repeated split metrics (percentages):")
display(repeat_display.round(2))
print("GroupKFold fold metrics (percentages):")
display(fold_display.round(2))
print("Repeated split summary (mean, sample SD, minimum and maximum; percentages):")
display(repeat_summary_display.round(2))
print("GroupKFold summary (mean, sample SD, minimum and maximum; percentages):")
display(fold_summary_display.round(2))
print("Repeated split continent false-negative / false-positive counts:")
display(repeat_continent_errors)
print("GroupKFold continent false-negative / false-positive counts:")
display(fold_continent_errors)
print(f"Repeated logistic-regression accuracy: {repeat_accuracy_mean:.2f}% ± {repeat_accuracy_sd:.2f}%")
print(f"GroupKFold logistic-regression accuracy: {fold_accuracy_mean:.2f}% ± {fold_accuracy_sd:.2f}%")
print("Country overlap maximum — repeats:", repeat_metrics["country_overlap"].max())
print("Country overlap maximum — GroupKFold:", fold_metrics["country_overlap"].max())

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.boxplot(
    [repeat_metrics["model_accuracy"] * 100, repeat_metrics["baseline_accuracy"] * 100],
    tick_labels=["Logistic regression", "Majority baseline"],
    showmeans=True,
)
ax.scatter(np.repeat(1, len(repeat_metrics)),
           repeat_metrics["model_accuracy"] * 100, alpha=0.75)
ax.scatter(np.repeat(2, len(repeat_metrics)),
           repeat_metrics["baseline_accuracy"] * 100, alpha=0.75)
ax.set_ylabel("Held-out accuracy (%)")
ax.set_title("Accuracy across 10 country-grouped repeats (seeds 42–51)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "phase6_accuracy_boxplot.png", dpi=180, bbox_inches="tight")
plt.show()